## Lecture Notes: Generators in Python
### 1. Introduction and Definition

Generators are often considered a difficult or confusing topic in Python programming.

*   **Definition:** Python Generators are a **simple way of creating Iterators**.
*   **Purpose:** They simplify the process of creating iterators. Previously, creating a simple iterator (like a custom `range` function) required defining complex classes: one for the iterable and one for the iterator, implementing both the `__iter__` and `__next__` functions. Generators offer a modern, simpler approach.

### 2. The Necessity of Iterators and Generators (Memory Efficiency)

Iterators (and thus generators) solve the problem of memory limitations when dealing with large datasets.

*   **The Problem with Lists:** If you need to process a large number of items (e.g., 100,000 numbers and their squares), storing them all in a list requires allocating a large amount of memory. If the list contains 10 million or more items, the memory requirements become prohibitive.
*   **Memory Efficiency:** Generators are memory efficient. An iterator (like the built-in `range` function) works by **only keeping one item in memory at a time**.
    *   It retrieves one item, performs the required operation (e.g., printing its square), then removes it from memory before fetching the next item.
    *   Using `range` up to 100,000 items, the memory size remains very small (e.g., 8 bytes), regardless of whether you increase the range to 10 million.
*   **Large and Infinite Data:** This efficiency allows you to work with **infinitely large numbers/streams** or huge data sets, as the memory footprint is constant.

### 3. Generator Syntax and Execution

Generators are essentially specialized functions with one major syntactic difference.

| Feature | Normal Python Function | Python Generator Function |
| :--- | :--- | :--- |
| **Statement** | `return` | **`yield`** |
| **Output on Call** | Returns the output value immediately | Returns a **Generator Object** |

#### A. Execution Flow

1.  **Function Call:** When the generator function is initially called (e.g., `gen = gen_demo()`), **no execution occurs** within the function body; instead, a Generator Object is created and stored.
2.  **Item Retrieval (`next` or Loop):** Execution begins only when the `next()` function is called on the Generator Object, or when it is iterated over using a `for` loop.
3.  **`yield` Statement:** Each time `next()` is called, the function executes code until it reaches a `yield` statement. The value specified in the `yield` statement is returned to the caller.
4.  **Completion:** Once all `yield` statements have been executed, running `next()` again results in a `StopIteration` error.

#### B. The Difference Between `yield` and `return` (State Preservation)

The fundamental difference between generators and normal functions lies in how they handle state.

*   **Normal Function (`return`):** Executes its entire logic, returns the output, finishes its work, and is then removed from memory.
*   **Generator Function (`yield`):**
    *   **Partial Execution and Pause:** The generator executes its work partially. When it hits a `yield` statement, it pauses execution and temporarily exits memory.
    *   **State Preservation:** Crucially, the generator remembers its state, including the values of its local variables and the exact line of code it executed last.
    *   **Resumption:** When the generator is called again (via `next`), it **restarts execution exactly from where it paused**, using the preserved state.

### 4. Generator Examples and Expressions

#### A. Custom Range Function

The generator approach simplifies creating functions that behave like `range`:

*   A custom range generator function (e.g., `my_range(start, end)`) iterates using a `for` loop and simply uses `yield` for the value of the iterator (`i`). This accomplishes the same goal as the extensive class-based code required for traditional iterators.

#### B. Practical Use Case: Handling Large Data

Generators are highly beneficial in applications involving massive data, such as Deep Learning:

*   **Scenario:** When a dataset (e.g., thousands of high-resolution images) is several gigabytes in size (e.g., 30 GB), it cannot be loaded into limited memory (e.g., 8 GB RAM) all at once.
*   **Generator Solution:** A generator function can be written to accept a folder path, access all files, and then loop through them. It **loads only one image at a time**, converts it to a NumPy array, and uses `yield` to pass it to the main program for processing or model training.
*   This approach ensures that regardless of whether there are 4,000 images or 40 million, only a single image is held in memory at any given time.

#### C. Generator Expressions

Generator expressions provide a very concise, single-line way to create a generator without defining a separate function.

*   **Syntax:** They closely resemble list comprehensions, but use **small brackets `()`** instead of square brackets `[]`.
*   They are useful when the logic is simple enough to be written on one line.

### 5. Benefits of Using Generators

1.  **Ease of Implementation (Simplified Code):** Generators make creating iterators dramatically simpler ("a piece of cake") compared to the complex class structure previously required.
2.  **Memory Efficiency:** They save memory because they only load and process items one at a time, making them essential for working with large or massive datasets.
3.  **Handling Infinite Streams:** Generators enable computations involving potentially infinite sequences (e.g., generating all possible even numbers) because they never attempt to store the entire sequence in memory.
4.  **Chaining/Connecting:** Multiple generators can be logically linked together to perform complex tasks. For example, one generator can produce a Fibonacci sequence, and a subsequent generator can take those results, square them, and then the final result can be summed up.

In [ ]:
# a generator fn to load only one batch of images in memory in one time
def load_images_from_directory(directory):
  for batch_no in range(len(os.listdir(directory))//BATCH_SIZE):
    result=[]
    for img in sorted(os.listdir(directory))[batch_no*BATCH_SIZE:BATCH_SIZE*(batch_no+1)]:
      img_path=os.path.join(directory,img)
      img=cv2.imread(img_path)
      img=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
      # reshaping to -1 to 1
      img=img/127.5-1
      result.append(img)
    yield np.array(result)